# Models

## XgBoost

In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix

# === Load dataset from parquet ===
file_path = r"C:\Users\jeeva\Documents\FedSentry\data\processed\xgboost_dataset.parquet"
df = pd.read_parquet(file_path)

# === Encode categorical features ===
categorical_features = ["HTTP", "HTTPS", "DNS", "Telnet", "SMTP", "SSH", "IRC",
                        "TCP", "UDP", "DHCP", "ARP", "ICMP", "IGMP", "IPv", "LLC"]

for col in categorical_features:
    df[col] = LabelEncoder().fit_transform(df[col].astype(str))

# === Feature engineering ===
df["flag_intensity"] = (df["fin_flag_number"] + df["syn_flag_number"] +
                        df["rst_flag_number"] + df["psh_flag_number"] +
                        df["ack_flag_number"] + df["ece_flag_number"] +
                        df["cwr_flag_number"])

df["packet_density"] = df["Tot size"] / (df["IAT"] + 1e-6)   # avoid division by zero
df["handshake_score"] = df["ack_count"] + df["syn_count"] + df["fin_count"] + df["rst_count"]

# === Define features and labels ===
X = df.drop(columns=["Label"])
y = df["Label"]

# Encode labels
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# === Train/Validation/Test split ===
X_train, X_temp, y_train, y_temp = train_test_split(X, y_encoded, test_size=0.3, stratify=y_encoded, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42)

# === Compute class weights ===
class_counts = np.bincount(y_encoded)
total_samples = len(y_encoded)
num_classes = len(class_counts)
class_weights = {i: total_samples / (num_classes * count) for i, count in enumerate(class_counts)}

# Map weights to training labels
sample_weights = np.array([class_weights[label] for label in y_train])

# === Configure XGBoost hyperparameters ===
xgb_model = XGBClassifier(
    objective="multi:softprob",
    num_class=num_classes,
    max_depth=8,
    learning_rate=0.1,
    n_estimators=500,
    subsample=0.8,
    colsample_bytree=0.8,
    gamma=1,
    reg_alpha=0.1,
    reg_lambda=1,
    random_state=42,
    n_jobs=-1
)

# === Train model with class weights ===
xgb_model.fit(X_train, y_train, sample_weight=sample_weights,
              eval_set=[(X_val, y_val)], eval_metric="mlogloss", verbose=True)

# === Evaluate on test set ===
y_pred = xgb_model.predict(X_test)
print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=label_encoder.classes_))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))


ArrowMemoryError: malloc of size 8388608 failed